In [11]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 런타임 유형을 GPU(T4 이상)로 변경해 주세요!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4


In [12]:
!pip install -q transformers safetensors soundfile tqdm pandas scikit-learn

In [13]:
import os
import zipfile
from tqdm import tqdm

os.makedirs('/content/data', exist_ok=True)

# 방금 모아두신 내 드라이브의 DCC_Data 폴더 경로
zip_dir = '/content/drive/MyDrive/DCC_Data'
if not os.path.exists(zip_dir):
    # 혹시 다른 경로에 있을 경우 대비 탐색
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'DCC_Data' in dirs:
            zip_dir = os.path.join(root, 'DCC_Data')
            break

print("zip 폴더 위치:", zip_dir)

zip_files = sorted([
    os.path.join(zip_dir, f) for f in os.listdir(zip_dir)
    if f.endswith('.zip')
])

print(f"\n발견된 zip 파일 총 {len(zip_files)}개:")
for z in zip_files:
    print(" -", os.path.basename(z))

# 13개 zip 순차 고속 압축 해제
for zp in tqdm(zip_files, desc="전체 데이터 고속 압축 해제"):
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall('/content/data')

print("🎉 1~13번 전체 압축 해제 완료!")

zip 폴더 위치: /content/drive/MyDrive/DCC_Data

발견된 zip 파일 총 14개:
 - 대학부 데이터-20260830T163301Z-1-001.zip
 - 대학부 데이터-20260830T163301Z-1-002.zip
 - 대학부 데이터-20260830T163301Z-1-003.zip
 - 대학부 데이터-20260830T163301Z-1-004 (1).zip
 - 대학부 데이터-20260830T163301Z-1-004.zip
 - 대학부 데이터-20260830T163301Z-1-005.zip
 - 대학부 데이터-20260830T163301Z-1-006.zip
 - 대학부 데이터-20260830T163301Z-1-007.zip
 - 대학부 데이터-20260830T163301Z-1-008.zip
 - 대학부 데이터-20260830T163301Z-1-009.zip
 - 대학부 데이터-20260830T163301Z-1-010.zip
 - 대학부 데이터-20260830T163301Z-1-011.zip
 - 대학부 데이터-20260830T163301Z-1-012.zip
 - 대학부 데이터-20260830T163301Z-1-013.zip


전체 데이터 고속 압축 해제: 100%|██████████| 14/14 [12:23<00:00, 53.12s/it]

🎉 1~13번 전체 압축 해제 완료!


In [17]:
import os
import json
import pandas as pd
from tqdm import tqdm

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]

# 라벨 폴더 탐색
label_dir = None
for root, dirs, files in os.walk('/content/data'):
    if 'TL_서울_구급' in root or ('라벨링데이터' in root and any(f.endswith('.json') for f in files)):
        label_dir = root
        break

if not label_dir:
    raise FileNotFoundError("라벨 json 폴더를 찾을 수 없습니다. 경로를 확인하세요.")
print("라벨 디렉토리:", label_dir)

json_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.json')])
print(f"총 유입된 원본 통화 수: {len(json_files):,}건")

augmented_rows = []
for fn in tqdm(json_files, desc="화자 태그 Prefix 데이터 증강"):
    fp = os.path.join(label_dir, fn)
    with open(fp, 'r', encoding='utf-8') as f:
        d = json.load(f)

    raw = d.get('symptom') or []
    valid = [s for s in TARGET_SYMPTOMS if s in raw]
    if not valid:
        continue  # 빈 증상 샘플 제외

    utts = d.get('utterances') or []
    utts.sort(key=lambda u: int(u.get('startAt', 0)))
    if not utts:
        continue

    def format_utt(u):
        spk = int(u.get('speaker', -1))
        tag = "[대원] " if spk == 0 else ("[신고자] " if spk == 1 else "")
        txt = u.get('text', '').strip()
        return f"{tag}{txt}" if txt else ""

    # 검증된 최적 슬라이싱 구간 (3, 5, 8, 12, 전체)
    cutoffs = [3, 5, 8, 12, len(utts)]
    seen_cutoffs = set()

    for c in cutoffs:
        c = min(c, len(utts))
        if c in seen_cutoffs:
            continue
        seen_cutoffs.add(c)

        prefix_texts = [format_utt(u) for u in utts[:c]]
        prefix_text = " ".join(t for t in prefix_texts if t).strip()
        if not prefix_text:
            continue

        row = {'call_id': fn, 'cutoff_step': c, 'text': prefix_text}
        for s in TARGET_SYMPTOMS:
            row[s] = 1 if s in valid else 0
        augmented_rows.append(row)

df = pd.DataFrame(augmented_rows)
print(f"\n[데이터 증강 완료] 유효 통화: {len(json_files):,}건 -> 생성된 학습 샘플: {len(df):,}건")
df.to_csv('/content/augmented_train_speaker.csv', index=False, encoding='utf-8-sig')
print("✅ /content/augmented_train_speaker.csv 저장 완료!")

라벨 디렉토리: /content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급
총 유입된 원본 통화 수: 3,640건


화자 태그 Prefix 데이터 증강: 100%|██████████| 3640/3640 [00:01<00:00, 3364.55it/s]



[데이터 증강 완료] 유효 통화: 3,640건 -> 생성된 학습 샘플: 18,144건
✅ /content/augmented_train_speaker.csv 저장 완료!


In [19]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]

# 🚀 구어체·대화체 특화 사전학습 모델: KcELECTRA-base-v2022
MODEL_NAME = "beomi/KcELECTRA-base-v2022"

BATCH_SIZE = 32
EPOCHS = 1
LR = 3e-5
MAX_LEN = 512
POS_WEIGHT_POWER = 0.35  # RoBERTa Exp 2와 100% 동일 조건

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Testing Model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

# 화자 태그 증강 데이터셋 로드 (001번 데이터 기준)
df = pd.read_csv('/content/augmented_train_speaker.csv')

# 동일 시드(42) 분할로 완벽히 동일한 검증셋 보장
unique_calls = df['call_id'].unique()
train_calls, val_calls = train_test_split(unique_calls, test_size=0.15, random_state=42)

train_df = df[df['call_id'].isin(train_calls)].reset_index(drop=True)
val_df = df[df['call_id'].isin(val_calls)].reset_index(drop=True)
print(f"Train 통화: {len(train_calls):,}건 ({len(train_df):,}샘플) / Val 통화: {len(val_calls):,}건 ({len(val_df):,}샘플)")

class SymptomDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.texts = data['text'].tolist()
        self.labels = data[TARGET_SYMPTOMS].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_loader = DataLoader(SymptomDataset(train_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SymptomDataset(val_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

# 동일 pos_weight (p=0.35) 계산
num_pos = train_df[TARGET_SYMPTOMS].sum().values
num_neg = len(train_df) - num_pos
pos_weight = (num_neg / np.maximum(num_pos, 1)) ** POS_WEIGHT_POWER
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(TARGET_SYMPTOMS))
model.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

# 1. 학습 진행 (약 12~14분)
model.train()
total_train_loss = 0
for batch in tqdm(train_loader, desc="KcELECTRA Training (1 Epoch)"):
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    loss = criterion(outputs.logits, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()
    total_train_loss += loss.item()

print(f"\nFinal KcELECTRA Train Loss: {total_train_loss / len(train_loader):.4f}")

# 2. 로컬 저장 및 구글 드라이브 백업
save_dir = '/content/best_kcelectra_model'
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

drive_model_path = '/content/drive/MyDrive/DCC_best_kcelectra_model'
os.system(f"cp -r {save_dir} {drive_model_path}")
print(f"✅ 구글 드라이브 모델 백업 완료: {drive_model_path}")

# 3. 4-2용 KcELECTRA 확률표 추출 및 드라이브 저장
model.eval()
dump_rows = []
with torch.no_grad():
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="KcELECTRA 확률표 추출"):
        inputs = tokenizer(
            str(row['text']),
            max_length=MAX_LEN,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

        record = {'call_id': row['call_id'], 'step': row['cutoff_step']}
        for i, s in enumerate(TARGET_SYMPTOMS):
            record[s] = float(probs[i])
        dump_rows.append(record)

prob_df = pd.DataFrame(dump_rows)
prob_df.to_csv('/content/val_probabilities_kcelectra.csv', index=False, encoding='utf-8-sig')
os.system("cp /content/val_probabilities_kcelectra.csv /content/drive/MyDrive/DCC_val_probabilities_kcelectra.csv")
print("🎉 KcELECTRA 실험 완료! 구글 드라이브에 'DCC_best_kcelectra_model'이 저장되었습니다.")

Using device: cuda
Testing Model: beomi/KcELECTRA-base-v2022


config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Train 통화: 3,094건 (15,421샘플) / Val 통화: 546건 (2,723샘플)


... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 구글 드라이브 모델 백업 완료: /content/drive/MyDrive/DCC_best_kcelectra_model


KcELECTRA 확률표 추출: 100%|██████████| 2723/2723 [00:34<00:00, 77.92it/s]


🎉 KcELECTRA 실험 완료! 구글 드라이브에 'DCC_best_kcelectra_model'이 저장되었습니다.
